<a href="https://colab.research.google.com/github/welingtonsouza-senai/CodigoDosAlunos/blob/main/Semana5/Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

print("Baixando base global da COVID-19 (Pode demorar até 1 minuto.)...")
# Base de dados oficial da Our World in Data (Mais de 80MB de CSV)
url_covid = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"

# Lendo direto da web
df_covid = pd.read_csv(url_covid)

print(f"Base carregada com SUCESSO!")
print(f"Dimensões: {df_covid.shape[0]} linhas e {df_covid.shape[1]} colunas.")

# 1. Transformação Temporal (Convertendo texto para Datetime)
df_covid['date'] = pd.to_datetime(df_covid['date'])

# 2. Filtragem Extrema (Máscara Booleana)
# Queremos ver apenas o impacto no Brasil durante o ano crítico de 2021
mascara_br_2021 = (df_covid['location'] == 'Brazil') & (df_covid['date'].dt.year == 2021)
df_brasil = df_covid[mascara_br_2021]

print(f"\n--- Dados Filtrados: Brasil em 2021 ({df_brasil.shape[0]} dias) ---")
display(df_brasil[['date', 'new_cases', 'new_deaths']].head())

# 3. GroupBy de Alto Nível (Removendo Nulos antes de Agrupar)
# Qual foi a média de mortes diárias por continente?
# Primeiro jogamos fora as linhas onde o continente está vazio (NaN)
df_continentes = df_covid.dropna(subset=['continent'])

mortes_por_continente = df_continentes.groupby('continent')['new_deaths'].mean().sort_values(ascending=False)

print("\n--- Média de Mortes Diárias por Continente (Todo o Período) ---")
display(mortes_por_continente)

Baixando base global da COVID-19 (Pode demorar até 1 minuto.)...
Base carregada com SUCESSO!
Dimensões: 429435 linhas e 67 colunas.

--- Dados Filtrados: Brasil em 2021 (365 dias) ---


,date,new_cases,new_deaths
50596,2021-01-01,0.0,0.0
50597,2021-01-02,0.0,0.0
50598,2021-01-03,252018.0,4923.0
50599,2021-01-04,0.0,0.0
50600,2021-01-05,0.0,0.0



--- Média de Mortes Diárias por Continente (Todo o Período) ---


,new_deaths
continent,
South America,57.931257
Europe,24.971814
North America,24.355413
Asia,20.811376
Africa,2.715698
Oceania,0.822024


In [9]:
import pandas as pd
import requests

print("Consumindo a API Oficial do Prêmio Nobel...")
url_json = "https://api.nobelprize.org/v1/prize.json"
headers = {'User-Agent': 'Mozilla/5.0'}

resposta = requests.get(url_json, headers=headers)
dados_json = resposta.json()

# CORREÇÃO: Limpando a sujeira da API antes do Pandas!
# Filtramos apenas os prêmios onde a chave 'laureates' existe (excluindo os anos de Guerra Mundial)
premios_com_ganhadores = [premio for premio in dados_json['prizes'] if 'laureates' in premio]

# Agora o Pandas consegue achatar sem quebrar
df_nobel = pd.json_normalize(
    premios_com_ganhadores,
    record_path=['laureates'],
    meta=['year', 'category']
)

print(f"JSON Achadado! Temos {df_nobel.shape[0]} laureados na história.")

# Limpeza e Pivot
df_nobel['year'] = pd.to_numeric(df_nobel['year'])
df_moderno = df_nobel[df_nobel['year'] >= 2010]

matriz_nobel = df_moderno.pivot_table(
    values='id',
    index='category',
    columns='year',
    aggfunc='count',
    fill_value=0
)

print("\n--- Matriz Dinâmica: Distribuição de Prêmios (A partir de 2010) ---")
display(matriz_nobel)

Consumindo a API Oficial do Prêmio Nobel...
JSON Achadado! Temos 1026 laureados na história.

--- Matriz Dinâmica: Distribuição de Prêmios (A partir de 2010) ---


year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
category,,,,,,,,,,,,,,,,
chemistry,3,1,2,3,3,3,3,3,3,3,2,2,3,3,3,3
economics,3,2,2,3,1,1,2,1,2,3,2,3,3,1,3,3
literature,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
medicine,1,3,2,3,3,3,1,3,2,3,3,2,1,2,2,3
peace,1,3,1,1,2,1,1,1,2,1,1,2,3,1,1,1
physics,2,3,2,2,3,2,3,3,3,3,3,3,3,3,2,3


In [8]:
import pandas as pd
import sqlite3
import urllib.request
import os

print("Baixando Banco de Dados SQL completo (Chinook)...")

#  URL direta garantida e download seguro via urllib
url_sql = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
urllib.request.urlretrieve(url_sql, "chinook.sqlite")

# Criando a conexão com o arquivo do Banco de Dados baixado
conexao = sqlite3.connect("chinook.sqlite")

# Nomes das tabelas com a Primeira Letra Maiúscula (Padrão Chinook)
df_faturas = pd.read_sql_query("SELECT InvoiceId, CustomerId, InvoiceDate, Total FROM Invoice", conexao)
df_clientes = pd.read_sql_query("SELECT CustomerId, FirstName, Country FROM Customer", conexao)

# Fechar a conexão
conexao.close()

print(f"Banco lido com sucesso! Faturas: {df_faturas.shape[0]} | Clientes: {df_clientes.shape[0]}")

# O GRANDE MERGE (Cruzando faturas com o cadastro do cliente)
df_vendas_completas = pd.merge(
    left=df_faturas,
    right=df_clientes,
    on='CustomerId',
    how='inner'
)

# Agregando Valor: Descobrindo quais países dão mais lucro
rank_paises = df_vendas_completas.groupby('Country')['Total'].sum().sort_values(ascending=False).head(5)

print("\n--- TOP 5 Países que Mais Compram (Faturamento Total) ---")
display(rank_paises)

# Limpando o laboratório para não pesar o computador do aluno
os.remove("chinook.sqlite")

Baixando Banco de Dados SQL completo (Chinook)...
Banco lido com sucesso! Faturas: 412 | Clientes: 59

--- TOP 5 Países que Mais Compram (Faturamento Total) ---


,Total
Country,
USA,523.06
Canada,303.96
France,195.10
Brazil,190.10
Germany,156.48


In [13]:
import pandas as pd

print("Baixando base Excel do E-commerce Europeu...")
print("Notem como o Excel demora muito mais para carregar que o CSV, mesmo sendo um arquivo menor. Tenham paciência (pode levar 1 ou 2 minutos)!")

# Link direto para o Dataset público da UCI (Online Retail com +500 mil linhas)
url_excel = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

# Lendo direto da web (o Colab já possui o motor 'openpyxl' instalado por padrão para ler .xlsx)
df_varejo = pd.read_excel(url_excel)

print(f"Excel Carregado com Sucesso! Foram lidas {df_varejo.shape[0]} linhas.")
df_varejo.head()

# 1. Limpeza Pesada do Mundo Real
# No varejo, quando a quantidade (Quantity) é negativa, significa que o cliente devolveu o produto.
# Vamos filtrar para pegar APENAS as vendas reais (Quantity > 0) e clientes que tenham ID preenchido.
df_limpo = df_varejo[(df_varejo['Quantity'] > 0) & (df_varejo['CustomerID'].notna())].copy()

# 2. Engenharia de Features (Criando a Coluna de Faturamento)
# O dataset tem a Quantidade e o Preço Unitário, mas não o total. O Pandas faz isso com vetorização!
df_limpo['Faturamento_Total'] = df_limpo['Quantity'] * df_limpo['UnitPrice']

# 3. Análise: Quais os 5 países (excluindo o Reino Unido, que é a sede) que mais dão lucro?
# Filtramos fora o Reino Unido
df_exportacao = df_limpo[df_limpo['Country'] != 'United Kingdom']

# Agrupamos por país e somamos o faturamento
top_paises = df_exportacao.groupby('Country')['Faturamento_Total'].sum().sort_values(ascending=False).head(5)

print("\n--- TOP 5 Países em Exportação (Faturamento em Libras £) ---")
display(top_paises)

# Mostrando um detalhe da tabela pronta para Machine Learning
print("\n--- Amostra da Tabela Enriquecida ---")
display(df_limpo[['InvoiceNo', 'Description', 'Quantity', 'UnitPrice', 'Faturamento_Total', 'Country']].head())

Baixando base Excel do E-commerce Europeu...
Notem como o Excel demora muito mais para carregar que o CSV, mesmo sendo um arquivo menor. Tenham paciência (pode levar 1 ou 2 minutos)!
Excel Carregado com Sucesso! Foram lidas 541909 linhas.

--- TOP 5 Países em Exportação (Faturamento em Libras £) ---


,Faturamento_Total
Country,
Netherlands,285446.34
EIRE,265545.90
Germany,228867.14
France,209024.05
Australia,138521.31



--- Amostra da Tabela Enriquecida ---


,InvoiceNo,Description,Quantity,UnitPrice,Faturamento_Total,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,15.30,United Kingdom
1,536365,WHITE METAL LANTERN,6,3.39,20.34,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,2.75,22.00,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,20.34,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,20.34,United Kingdom
